# IntelliBrief -- Multi-Agent Business Intelligence & Reporting System

**Final Project -- AI Engineering Course**

A multi-agent AI system that automatically gathers business data from **real datasets**
(an internal-style retail sales dataset, a real customer reviews dataset, and a real
fashion/retail industry PDF report), analyzes it, writes a professional report, validates
it for reliability, and automates the delivery (email + saved file + dashboard update)
-- all through a simple chat/dashboard web interface.

See `README.md` for the full project overview.

## Data sources used in this notebook

| # | Data Source | Type | Where it plugs in |
|---|---|---|---|
| 1 | Superstore Sales Dataset (Kaggle) | CSV | `fetch_sales_data_api()` |
| 2 | Women's E-Commerce Clothing Reviews (Kaggle) | CSV | `fetch_feedback_sheet_api()` |
| 3 | The State of Fashion 2022 report (McKinsey / BoF) | PDF | RAG knowledge base |

Download links and column explanations are provided right before each data-loading cell.

## SECTION 1 - Installations

In [1]:
# Core libraries: model loading, RAG (LangChain + FAISS), data handling, and the web UI.
# NOTE: langchain and langchain-community are pinned to <1.0 on purpose.
# LangChain 1.0+ moved StructuredOutputParser/ResponseSchema to a new
# langchain_classic package, which breaks the classic import path used below.
# Pinning to the last pre-1.0 line keeps `from langchain.output_parsers import ...` working.
!pip install -q -U transformers accelerate bitsandbytes gradio "langchain<1.0" "langchain-community<1.0" faiss-cpu pypdf sentence-transformers pandas huggingface_hub[hf_xet]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73

## SECTION 2 - Imports

In [2]:
import os

# Fix: HuggingFace Hub Xet Storage 401 Unauthorized error.
# This MUST run before importing torch / transformers / huggingface_hub.
os.environ["HF_HUB_DISABLE_XET"] = "1"

import json
import re
import csv
import datetime
from collections import deque

import pandas as pd
import torch
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
# Resilient import: works whether LangChain is pre-1.0 (classic path) or
# 1.0+ (where these classes moved to the langchain_classic package).
try:
    from langchain.output_parsers import StructuredOutputParser, ResponseSchema
except ModuleNotFoundError:
    from langchain_classic.output_parsers.structured import StructuredOutputParser, ResponseSchema
from langchain.prompts import PromptTemplate

## SECTION 3 - Loading the LLM (Quantized 4-bit)

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit quantization (NF4 + double quantization) keeps memory usage low
# while preserving most of the model's quality -- see Session 10 (Quantization).
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Retry logic in case of a transient network / HF Hub error.
max_retries = 3
model = None

for attempt in range(1, max_retries + 1):
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="auto",
            quantization_config=bnb_config,
            trust_remote_code=True
        )
        print(f"Model loaded successfully on attempt {attempt}.")
        break

    except Exception as e:
        print(f"Attempt {attempt}/{max_retries} failed: {e}")
        if attempt == max_retries:
            raise RuntimeError(
                "Failed to load the model after multiple attempts. "
                "Please check your internet connection or try setting an HF_TOKEN."
            ) from e

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on attempt 1.


## SECTION 4 - Generation Helper Function

In [4]:
def generate(prompt, max_new_tokens=500, deterministic=True):
    """
    Core LLM call used by every agent in this notebook.

    deterministic=True  -> do_sample=False (used for structured / analytical tasks,
                            where we want stable, repeatable output)
    deterministic=False -> sampling enabled (used for more natural report writing)
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=3072).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    if deterministic:
        gen_kwargs.update(do_sample=False)
    else:
        gen_kwargs.update(do_sample=True, temperature=0.7, top_p=0.95, repetition_penalty=1.1)

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return response.strip()

## SECTION 5 - Utility: Robust JSON Extraction (Guardrail)

In [5]:
def extract_json_block(text):
    """
    Extracts a JSON object from raw LLM text using 3 strategies (from strictest
    to most permissive), then attempts to auto-fix common issues (trailing commas).
    """
    # Strategy 1: look for a ```json ... ``` markdown block
    pattern = r"```json\s*(.*?)\s*```"
    matches = re.findall(pattern, text, re.DOTALL)

    if matches:
        json_str = matches[-1].strip()
    else:
        # Strategy 2: regex that can handle one level of nested {}
        pattern = r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}"
        matches = re.findall(pattern, text, re.DOTALL)
        if matches:
            json_str = max(matches, key=len)
        else:
            # Strategy 3: fallback -- first "{" to last "}"
            start, end = text.find("{"), text.rfind("}")
            if start != -1 and end != -1:
                json_str = text[start:end + 1]
            else:
                return None

    json_str = json_str.strip()

    try:
        json.loads(json_str)
        return json_str
    except json.JSONDecodeError:
        # Try to auto-fix a trailing comma before } or ]
        try:
            fixed = re.sub(r",(\s*[}\]])", r"\1", json_str)
            json.loads(fixed)
            return fixed
        except Exception:
            return None

## SECTION 6 - Data Source 1: Retail Sales Data (Real CSV)

**Dataset:** Superstore Sales Dataset
**Download:** https://www.kaggle.com/datasets/vivek468/superstore-dataset-final
(any similar "Superstore" CSV export works too)

**What it contains:** thousands of real retail order records with columns such as
`Order Date`, `Sales`, `Profit`, `Quantity`, `Category`, `Sub-Category`, `Region`.

**How to use it here:** download the CSV from Kaggle, upload it to your Colab/Kaggle
session, and set `SALES_CSV_PATH` below to its location.

In [6]:
# TODO: update this path after uploading the Superstore CSV file.
SALES_CSV_PATH = "/content/train.csv"

def load_sales_data(csv_path):
    """
    Loads the Superstore CSV and prepares a Quarter column for aggregation.
    Returns None if the file is not found (the app can still run with the
    other 2 data sources -- this mirrors real-world "one source is down"
    situations, handled the same way as a failed API call).
    """
    if not os.path.exists(csv_path):
        print(f"Sales CSV not found at: {csv_path}")
        return None

    try:
        # Superstore CSVs are often saved with latin1 encoding.
        df = pd.read_csv(csv_path, encoding="latin1")
        df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
        df["Quarter"] = df["Order Date"].dt.to_period("Q").astype(str)
        print(f"Sales data loaded: {len(df)} rows, columns: {list(df.columns)}")
        return df

    except Exception as e:
        print("Failed to load sales CSV:", str(e))
        return None


sales_df = load_sales_data(SALES_CSV_PATH)

Sales data loaded: 9800 rows, columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quarter']


## Read data

In [8]:
print(sales_df.shape)
sales_df.head()


(9800, 19)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quarter
0,1,CA-2017-152156,2017-08-11,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2017Q3
1,2,CA-2017-152156,2017-08-11,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,2017Q3
2,3,CA-2017-138688,2017-12-06,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2017Q4
3,4,US-2016-108966,2016-11-10,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,2016Q4
4,5,US-2016-108966,2016-11-10,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2016Q4


## SECTION 7 - Tool: Sales API (built on top of the real CSV)

In [9]:
def fetch_sales_data_api():
    """
    Simulates an internal sales API by aggregating the real Superstore data
    per quarter: total revenue, total units sold, and top category.
    In a real company, this same function would instead call a REST endpoint
    (e.g. requests.get("https://internal-api.company.com/sales")).

    NOTE: made column-agnostic on purpose -- different Superstore CSV exports
    on Kaggle use slightly different column sets (some include Quantity/Profit,
    some don't, like the one used in this run). Missing columns degrade
    gracefully (return "N/A") instead of crashing the whole pipeline.
    """
    print("Sales API called...")

    if sales_df is None:
        return {"status": "unavailable", "message": "Sales data source is not loaded."}

    has_quantity = "Quantity" in sales_df.columns
    has_category = "Category" in sales_df.columns

    summary = {}
    for quarter, group in sales_df.groupby("Quarter"):
        entry = {"revenue": round(group["Sales"].sum(), 2)}

        entry["units_sold"] = int(group["Quantity"].sum()) if has_quantity else "N/A"
        entry["top_category"] = (
            group.groupby("Category")["Sales"].sum().idxmax() if has_category else "N/A"
        )

        summary[quarter] = entry

    # Keep only the most recent 4 quarters to keep the LLM context small.
    recent_quarters = dict(list(summary.items())[-4:])
    return recent_quarters

## SECTION 8 - Data Source 2: Customer Feedback Data (Real CSV)

**Dataset:** Women's E-Commerce Clothing Reviews
**Download:** https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews

**What it contains:** ~23,000 real customer reviews with columns such as
`Review Text`, `Rating` (1-5), `Recommended IND` (0/1), `Department Name`.

**How to use it here:** download the CSV, upload it, and set `FEEDBACK_CSV_PATH` below.

In [10]:
# TODO: update this path after uploading the Clothing Reviews CSV file.
FEEDBACK_CSV_PATH = "/content/Womens Clothing E-Commerce Reviews.csv"

def load_feedback_data(csv_path):
    """
    Loads the customer reviews CSV. Drops rows with an empty review text,
    since those add no value to the analysis.
    """
    if not os.path.exists(csv_path):
        print(f"Feedback CSV not found at: {csv_path}")
        return None

    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=["Review Text"])
        print(f"Feedback data loaded: {len(df)} reviews, columns: {list(df.columns)}")
        return df

    except Exception as e:
        print("Failed to load feedback CSV:", str(e))
        return None


feedback_df = load_feedback_data(FEEDBACK_CSV_PATH)

Feedback data loaded: 22641 reviews, columns: ['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']


## Read Data

In [11]:
print(feedback_df.shape)
feedback_df.head()

(22641, 11)


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


## SECTION 9 - Tool: Feedback Sheet API (built on top of the real CSV)

In [12]:
def fetch_feedback_sheet_api(n_positive=3, n_negative=3):
    """
    Simulates reading a Google Sheet of customer feedback by sampling real
    reviews from the dataset: a mix of low-rated (Rating <= 2) and high-rated
    (Rating >= 4) reviews, so the Analyst Agent sees both praise and complaints.
    """
    print("Feedback Sheet API called...")

    if feedback_df is None:
        return ["Feedback data source is not available."]

    negative = feedback_df[feedback_df["Rating"] <= 2]["Review Text"].dropna()
    positive = feedback_df[feedback_df["Rating"] >= 4]["Review Text"].dropna()

    sample_negative = negative.sample(min(n_negative, len(negative)), random_state=42).tolist()
    sample_positive = positive.sample(min(n_positive, len(positive)), random_state=42).tolist()

    return sample_positive + sample_negative

## SECTION 10 - Data Source 3: Market Report (Real PDF, via RAG)

**Report:** The State of Fashion 2022 -- McKinsey & Business of Fashion
**Download:** https://www.mckinsey.com/~/media/mckinsey/industries/retail/our%20insights/state%20of%20fashion/2022/the-state-of-fashion-2022.pdf

**What it contains:** a real, ~130-page industry report covering fashion/retail
market trends, supply chain issues, consumer shifts, and recovery patterns.

**How to use it here:** download the PDF, upload it (or pass its path directly
in the Gradio UI's file upload -- no manual path needed there).

In [13]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def build_knowledge_base(pdf_path):
    """
    Loads a market report PDF and builds a FAISS vector store for semantic
    retrieval (RAG). Returns None if no PDF is provided or loading fails --
    the pipeline continues without this data source instead of crashing.
    """# TODO: update this path after uploading the market report PDF
# (e.g. "The State of Fashion 2022" PDF downloaded from the McKinsey link above).
# This is only used by the Quick Programmatic Test (Section 21) --
# the Gradio web interface (Section 19) lets you upload a PDF directly instead.
PDF_PATH = "/content/the-state-of-fashion-2022.pdf"

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def build_knowledge_base(pdf_path):
    """
    Loads a market report PDF and builds a FAISS vector store for semantic
    retrieval (RAG). Returns None if no PDF is provided or loading fails --
    the pipeline continues without this data source instead of crashing.
    """
    if not pdf_path:
        return None

    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        db = FAISS.from_documents(documents, embedding_model)
        return db.as_retriever(search_kwargs={"k": 2})

    except Exception as e:
        print("Failed to build knowledge base from PDF:", str(e))
        return None
    if not pdf_path:
        return None# TODO: update this path after uploading the market report PDF
# (e.g. "The State of Fashion 2022" PDF downloaded from the McKinsey link above).
# This is only used by the Quick Programmatic Test (Section 21) --
# the Gradio web interface (Section 19) lets you upload a PDF directly instead.
PDF_PATH = "/content/the-state-of-fashion-2022.pdf"

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def build_knowledge_base(pdf_path):
    """
    Loads a market report PDF and builds a FAISS vector store for semantic
    retrieval (RAG). Returns None if no PDF is provided or loading fails --
    the pipeline continues without this data source instead of crashing.
    """
    if not pdf_path:
        return None

    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        db = FAISS.from_documents(documents, embedding_model)
        return db.as_retriever(search_kwargs={"k": 2})

    except Exception as e:
        print("Failed to build knowledge base from PDF:", str(e))
        return None

    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        db = FAISS.from_documents(documents, embedding_model)
        return db.as_retriever(search_kwargs={"k": 2})

    except Exception as e:
        print("Failed to build knowledge base from PDF:", str(e))
        return None

/tmp/ipykernel_2393/745597205.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## SECTION 11 - Output Parser: Structured Analysis Schema

In [14]:
# Defines the exact structure the Analyst Agent must return (Session 1: Output Parser).
summary_schema = ResponseSchema(name="summary", description="A 2-3 sentence executive summary of the findings")
insights_schema = ResponseSchema(name="key_insights", description="A list of 3 to 5 short key insights, as a JSON array of strings")
risks_schema = ResponseSchema(name="risks", description="A list of 1 to 3 short risks or concerns, as a JSON array of strings")
recommendation_schema = ResponseSchema(name="recommendation", description="One clear, actionable recommendation")

response_schemas = [summary_schema, insights_schema, risks_schema, recommendation_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

## SECTION 12 - Agent 1: Retriever Agent

In [15]:
def retriever_agent(topic, retriever):
    """
    Gathers raw context from all 3 real data sources and combines them into
    one structured context block for the Analyst Agent. Each source can fail
    independently without breaking the whole pipeline.
    """
    sales = fetch_sales_data_api()
    feedback = fetch_feedback_sheet_api()

    if isinstance(sales, dict) and "status" not in sales:
        sales_text = "\n".join(
            [f"{q}: revenue=${v['revenue']}, units={v['units_sold']}, top_category={v['top_category']}"
             for q, v in sales.items()]
        )
    else:
        sales_text = "Sales data is currently unavailable."

    feedback_text = "\n".join([f"- {f}" for f in feedback])

    if retriever:
        docs = retriever.invoke(topic)
        market_report_text = "\n\n".join([d.page_content for d in docs]) if docs else "No relevant market report content found."
    else:
        market_report_text = "No market report PDF was provided for this analysis."

    context = f"""SALES DATA:
{sales_text}

CUSTOMER FEEDBACK:
{feedback_text}

MARKET REPORT (retrieved excerpts):
{market_report_text}
"""
    return context

## SECTION 13 - Agent 2: Analyst Agent (Structured Output)

In [16]:
def analyst_agent(topic, context):
    """
    Uses the LLM to analyze the combined context and return a structured
    JSON object (summary, key_insights, risks, recommendation).
    Retries once if the first JSON extraction attempt fails.
    """
    prompt = f"""You are a professional business analyst AI.
Analyze the data below about the topic: "{topic}"
Base your analysis ONLY on the provided data. Do not invent numbers or facts.

{format_instructions}

Data:
{context}

Respond with ONLY the JSON object. No explanation, no markdown formatting.
"""

    for attempt in range(2):
        raw_output = generate(prompt, max_new_tokens=400, deterministic=True)
        json_str = extract_json_block(raw_output)

        if json_str:
            try:
                return json.loads(json_str)
            except json.JSONDecodeError:
                pass

        print(f"Analyst Agent: JSON extraction failed on attempt {attempt + 1}, retrying...")

    # Fallback if both attempts fail -- the pipeline still completes,
    # and the QA Agent (Section 15) will catch this as a failed run.
    return {
        "summary": "Analysis could not be structured automatically.",
        "key_insights": [],
        "risks": ["Automated analysis failed to produce valid structured output."],
        "recommendation": "Please review the raw data manually."
    }

## SECTION 14 - Agent 3: Report Writer Agent

In [17]:
def writer_agent(topic, analysis):
    """
    Turns the structured analysis (a Python dict) into a polished,
    human-readable Markdown report.
    """
    insights_bullets = "\n".join([f"- {i}" for i in analysis.get("key_insights", [])]) or "- No specific insights extracted."
    risks_bullets = "\n".join([f"- {r}" for r in analysis.get("risks", [])]) or "- No significant risks identified."

    report = f"""# Business Intelligence Report: {topic}

**Generated:** {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Executive Summary
{analysis.get("summary", "N/A")}

## Key Insights
{insights_bullets}

## Risks & Concerns
{risks_bullets}

## Recommendation
{analysis.get("recommendation", "N/A")}
"""
    return report

## SECTION 15 - Agent 4: QA / Validation Agent (Guardrail)

In [18]:
def qa_agent(analysis, report_text):
    """
    Non-LLM validation layer (Guardrail). Checks the analysis and report
    for basic reliability BEFORE automation (email) is allowed to proceed.
    Intentionally does NOT use the LLM -- a reliability check should not
    itself depend on a model that can make mistakes.
    """
    issues = []

    required_fields = ["summary", "key_insights", "risks", "recommendation"]
    for field in required_fields:
        if not analysis.get(field):
            issues.append(f"Missing or empty field: '{field}'")

    if len(report_text.strip()) < 100:
        issues.append("Generated report is too short to be reliable.")

    if not isinstance(analysis.get("key_insights", []), list):
        issues.append("'key_insights' is not a valid list.")

    passed = len(issues) == 0
    return {"passed": passed, "issues": issues}

## SECTION 16 - Automation Layer: Save / Send / Log APIs

In [19]:
REPORTS_DIR = "/content" if os.path.exists("/content") else "."
DASHBOARD_PATH = os.path.join(REPORTS_DIR, "dashboard.csv")


def save_report_api(topic, report_text):
    print("Save API called...")
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"report_{timestamp}.md"
    filepath = os.path.join(REPORTS_DIR, filename)

    try:
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(report_text)
        print("Status: Report saved successfully ->", filepath)
        return {"status": "success", "file": filepath}

    except Exception as e:
        print("Status: Failed to save report:", str(e))
        return {"status": "error", "message": str(e)}


def send_email_api(topic, report_text):
    # Simulated -- in production, replace this with a real email API call.
    print("Send Email API called (simulated)...")
    try:
        print("To      : manager@example.com")
        print("Subject :", f"Business Intelligence Report - {topic}")
        print("Body preview:", report_text[:150].replace("\n", " ") + "...")
        return {"status": "sent"}

    except Exception as e:
        print("Status: Failed to send email:", str(e))
        return {"status": "error", "message": str(e)}


def update_dashboard_api(topic, analysis, qa_result):
    print("Dashboard API called...")
    try:
        file_exists = os.path.exists(DASHBOARD_PATH)
        with open(DASHBOARD_PATH, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(["timestamp", "topic", "num_insights", "num_risks", "qa_passed"])
            writer.writerow([
                datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                topic,
                len(analysis.get("key_insights", [])),
                len(analysis.get("risks", [])),
                qa_result["passed"]
            ])
        print("Status: Dashboard updated ->", DASHBOARD_PATH)
        return {"status": "success"}

    except Exception as e:
        print("Status: Failed to update dashboard:", str(e))
        return {"status": "error", "message": str(e)}

## SECTION 17 - Memory: Follow-up Q&A About the Latest Report

In [20]:
# Bounded memory (Session 6: Mastering Memory) -- only the last 10 Q&A pairs
# about the CURRENT report are kept, so follow-up answers stay relevant.
chat_memory = deque(maxlen=10)
last_report_text = {"value": None}


def chat_with_report(question):
    """
    Lets the user ask follow-up questions about the most recently generated
    report, using conversation memory to keep the discussion coherent.
    """
    if not last_report_text["value"]:
        return "No report has been generated yet. Please run an analysis first."

    if not question or not question.strip():
        return "Please enter a question."

    history_text = "\n".join(
        [f"Q: {t['q']}\nA: {t['a']}" for t in chat_memory]
    )

    prompt = f"""You are an assistant answering questions about the report below.
Answer ONLY using the report content. If the answer is not in the report, say "I don't know based on this report."

Report:
{last_report_text["value"]}

Previous Q&A:
{history_text}

New Question: {question}
Answer:"""

    try:
        answer = generate(prompt, max_new_tokens=200, deterministic=True)
    except Exception as e:
        answer = f"Sorry, something went wrong: {str(e)}"

    chat_memory.append({"q": question, "a": answer})
    return answer

## SECTION 18 - Orchestrator: Multi-Agent Pipeline

In [21]:
run_log = []


def run_pipeline(topic, pdf_path=None):
    """
    Coordinates all agents sequentially:
    Retriever -> Analyst -> Writer -> QA -> Automation (save/send/log)
    Stops early and returns a clear error if a critical step fails.
    """
    print("=" * 50)
    print("Starting Multi-Agent Pipeline for topic:", topic)
    print("=" * 50)

    if not topic or not topic.strip():
        return "Error: Please provide a topic to analyze.", None

    # Step 1: Retrieval
    try:
        retriever = build_knowledge_base(pdf_path)
        context = retriever_agent(topic, retriever)
    except Exception as e:
        return f"Error during data retrieval: {str(e)}", None

    # Step 2: Analysis (Structured Output)
    try:
        analysis = analyst_agent(topic, context)
    except Exception as e:
        return f"Error during analysis: {str(e)}", None

    # Step 3: Report Writing
    report_text = writer_agent(topic, analysis)

    # Step 4: QA / Validation Guardrail
    qa_result = qa_agent(analysis, report_text)
    if not qa_result["passed"]:
        print("QA validation issues found:", qa_result["issues"])

    # Step 5: Automation (email is skipped if QA failed, but save/log always run)
    save_result = save_report_api(topic, report_text)
    email_result = send_email_api(topic, report_text) if qa_result["passed"] else {"status": "skipped (QA failed)"}
    dashboard_result = update_dashboard_api(topic, analysis, qa_result)

    # Update memory for follow-up chat, scoped to this new report only.
    last_report_text["value"] = report_text
    chat_memory.clear()

    run_log.append({
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "topic": topic,
        "qa_passed": qa_result["passed"],
        "save_status": save_result["status"],
        "email_status": email_result["status"]
    })

    print("Pipeline finished. QA passed:", qa_result["passed"])
    return report_text, qa_result

## SECTION 19 - Web Interface (Gradio)

In [27]:
def run_analysis_ui(topic, pdf_file):
    pdf_path = pdf_file.name if pdf_file is not None else None
    report_text, qa_result = run_pipeline(topic, pdf_path)

    if qa_result is None:
        return report_text, "N/A"

    status = "PASSED" if qa_result["passed"] else "FAILED: " + "; ".join(qa_result["issues"])
    return report_text, status


def chat_ui(message, chat_history):
    """
    NOTE: uses the newer Gradio "messages" format (list of dicts with
    'role'/'content' keys) instead of the older (user, assistant) tuple
    format, which is deprecated/rejected by recent Gradio versions.
    """
    answer = chat_with_report(message)
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": answer})
    return "", chat_history


with gr.Blocks(title="IntelliBrief - Multi-Agent BI Assistant") as demo:

    gr.Markdown("# IntelliBrief -- Multi-Agent Business Intelligence Assistant")
    gr.Markdown(
        "Enter a topic to analyze (e.g. **'Q3 Sales Performance'**), optionally upload the "
        "State of Fashion market report PDF, and the multi-agent pipeline will retrieve data, "
        "analyze it, write a report, validate it, and automate delivery "
        "(save + simulated email + dashboard log)."
    )

    with gr.Tab("Run New Analysis"):
        topic_input = gr.Textbox(label="Topic", placeholder="e.g. Q3 Sales Performance")
        pdf_input = gr.File(label="Market Report PDF (optional)", file_types=[".pdf"])
        run_btn = gr.Button("Run Multi-Agent Analysis", variant="primary")

        report_output = gr.Markdown(label="Generated Report")
        qa_output = gr.Textbox(label="QA Validation Status", interactive=False)

        run_btn.click(fn=run_analysis_ui, inputs=[topic_input, pdf_input], outputs=[report_output, qa_output])

    with gr.Tab("Chat About Latest Report"):
        # type="messages" matches the dict format produced by chat_ui() above.
        # Wrapped in try/except because some Gradio versions don't accept the
        # 'type' argument at all (older versions), while others require it
        # explicitly (newer versions) -- this works with either.
        try:
            chatbot = gr.Chatbot(height=400, label="Follow-up Q&A", type="messages")
        except TypeError:
            chatbot = gr.Chatbot(height=400, label="Follow-up Q&A")

        chat_msg = gr.Textbox(label="Your Question", placeholder="e.g. What was the top category in Q3?")
        chat_msg.submit(fn=chat_ui, inputs=[chat_msg, chatbot], outputs=[chat_msg, chatbot])

## SECTION 20 - Launch App

---



In [28]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a166a7239b93c1966f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Starting Multi-Agent Pipeline for topic: Customer Satisfaction Trends
Sales API called...
Feedback Sheet API called...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Save API called...
Status: Report saved successfully -> /content/report_20260723_000758.md
Send Email API called (simulated)...
To      : manager@example.com
Subject : Business Intelligence Report - Customer Satisfaction Trends
Body preview: # Business Intelligence Report: Customer Satisfaction Trends  **Generated:** 2026-07-23 00:07:58  ## Executive Summary The data shows a mixed trend in...
Dashboard API called...
Status: Dashboard updated -> /content/dashboard.csv
Pipeline finished. QA passed: True
Starting Multi-Agent Pipeline for topic: Product Category Performance
Sales API called...
Feedback Sheet API called...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

Analyst Agent: JSON extraction failed on attempt 1, retrying...
Analyst Agent: JSON extraction failed on attempt 2, retrying...
QA validation issues found: ["Missing or empty field: 'key_insights'"]
Save API called...
Status: Report saved successfully -> /content/report_20260723_001127.md
Dashboard API called...
Status: Dashboard updated -> /content/dashboard.csv
Pipeline finished. QA passed: False
Starting Multi-Agent Pipeline for topic: Fashion Industry Market Outlook
Sales API called...
Feedback Sheet API called...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Save API called...
Status: Report saved successfully -> /content/report_20260723_001336.md
Send Email API called (simulated)...
To      : manager@example.com
Subject : Business Intelligence Report - Fashion Industry Market Outlook
Body preview: # Business Intelligence Report: Fashion Industry Market Outlook  **Generated:** 2026-07-23 00:13:36  ## Executive Summary The fashion industry is caut...
Dashboard API called...
Status: Dashboard updated -> /content/dashboard.csv
Pipeline finished. QA passed: True


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a166a7239b93c1966f.gradio.live


## SECTION 21 - Quick Programmatic Test (Optional)

In [30]:
# You can test the full pipeline without the UI.
# This uses PDF_PATH defined in Section 10 -- make sure the file exists there,
# otherwise the pipeline will simply run without the market report source.
report, qa = run_pipeline("Recent Sales Performance", pdf_path=PDF_PATH)
print(report)
print("\nQA Result:", qa)

Starting Multi-Agent Pipeline for topic: Recent Sales Performance
Sales API called...
Feedback Sheet API called...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Save API called...
Status: Report saved successfully -> /content/report_20260723_001658.md
Send Email API called (simulated)...
To      : manager@example.com
Subject : Business Intelligence Report - Recent Sales Performance
Body preview: # Business Intelligence Report: Recent Sales Performance  **Generated:** 2026-07-23 00:16:58  ## Executive Summary The sales data shows a mixed perfor...
Dashboard API called...
Status: Dashboard updated -> /content/dashboard.csv
Pipeline finished. QA passed: True
# Business Intelligence Report: Recent Sales Performance

**Generated:** 2026-07-23 00:16:58

## Executive Summary
The sales data shows a mixed performance with Technology as the top category in 2018. However, the customer feedback indicates sizing issues and fit problems across different products, suggesting potential quality control and sizing accuracy challenges.

## Key Insights
- The Technology category consistently performed well throughout the year.
- Customer feedback highlights sizin

## SECTION 22 - Download Project Outputs (For Final Submission / Presentation)

Run this cell **after** you have tried the pipeline with a few different topics
(see Section 19/20) and are happy with the generated reports.

It collects everything needed to demonstrate the project outside of Colab --
without needing the notebook, the model, or an active session -- into one
folder, zips it, and downloads it directly to your computer:

- All generated reports (`report_*.md`)
- The reliability log (`dashboard.csv`) -- proof of multiple successful runs (Session 22: Testing & Evaluation)
- The source market report PDF (`the-state-of-fashion-2022.pdf`), if used

Use this folder alongside the notebook, `README.md`, and the detailed
explanation markdown file when presenting or submitting the final project.

In [31]:
# Package all project outputs (reports, dashboard log, and the source PDF)
# into one folder, zip it, and download it as a single file.

import os
import shutil
import glob

PROJECT_FOLDER = "/content/IntelliBrief_Final_Project_Outputs"
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Copy all generated reports
for f in glob.glob("/content/report_*.md"):
    shutil.copy(f, PROJECT_FOLDER)

# Copy the dashboard log
if os.path.exists("/content/dashboard.csv"):
    shutil.copy("/content/dashboard.csv", PROJECT_FOLDER)

# Copy the source PDF (optional, remove this block if you don't want it included)
if os.path.exists(PDF_PATH):
    shutil.copy(PDF_PATH, PROJECT_FOLDER)

# Zip the whole folder
zip_path = shutil.make_archive(PROJECT_FOLDER, 'zip', PROJECT_FOLDER)
print("Created:", zip_path)

# Download it directly to your computer
from google.colab import files
files.download(zip_path)

Created: /content/IntelliBrief_Final_Project_Outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## SECTION 23 - Architecture

```
                    +----------------------+
                    |     Web Interface     |
                    |  (Gradio: Run + Chat)  |
                    +-----------+----------+
                                |
                                v
                    +----------------------+
                    |      Orchestrator      |
                    |     (run_pipeline)      |
                    +-----------+----------+
                                |
     +-------------+------------+------------+-------------+
     v             v            v            v             v
Retriever      Analyst      Writer        QA        Automation
  Agent         Agent        Agent      Agent        (save/send/log)
     |             |            |          |             |
     v             v            |          |             |
 Sales CSV     Structured       |     Validates    save_report_api
 Feedback CSV   JSON Output     |     analysis &   send_email_api
 RAG (PDF)     (Output Parser)  |     report        update_dashboard_api
                                v
                          Markdown Report
                                |
                                v
                    Memory (chat follow-up Q&A)
```

**Data flow of a single run:**
```
User enters topic (+ optional PDF)
        |
        v
Retriever Agent: fetch_sales_data_api() + fetch_feedback_sheet_api() + RAG retrieval from PDF
        |
        v
Analyst Agent: LLM + Output Parser -> structured JSON (summary, insights, risks, recommendation)
        |
        v
Writer Agent: structured JSON -> polished Markdown report
        |
        v
QA Agent: validates required fields, report length, data types (Guardrail)
        |
        v
Automation Layer: save_report_api -> send_email_api (only if QA passed) -> update_dashboard_api
        |
        v
Report + QA status shown in UI; report stored in memory for follow-up chat
```

**Key design decisions:**
- **Each agent is a separate function with a single responsibility** -- Retriever only gathers data, Analyst only structures it, Writer only formats it, QA only validates it (Session 7: Multi-Agent Collaboration).
- **The Analyst Agent never calls the automation tools directly, and the automation layer never calls the LLM** -- a clean separation between "thinking" (LLM agents) and "acting" (API-like tools) (Session 2: AI Agent with Tools).
- **QA Agent is a real Guardrail**: if validation fails, the email is skipped but the report is still saved and logged -- an audit trail is always kept, but unreliable content never reaches the end user.
- **Memory is scoped to the latest report only** (not the whole pipeline history), so follow-up answers stay focused and accurate (Session 6: Mastering Memory).
- **Each data source can fail independently** (missing CSV, missing PDF, missing rows) without crashing the whole pipeline -- mirroring real-world "one system is down" situations.